# 72 — Build LGBM features + train LambdaRank (Stage C)

Walks HF train conversations, splits sessions 80/20, runs the full
Stage A+B retrieval+reranker pipeline to get top-100 candidates per
music turn, then extracts the extended 28-feature vectors per
(turn, candidate) pair. Trains LightGBM LambdaRank on the result.

**Prereqs**: Stage A + Stage B done; merged BGE-M3 + CE on Hub;
BGE-M3-FT catalog pickle on Drive (notebook 70 cell 6).

**Wallclock**: ~4-6 hr on Blackwell (feature extraction is wRRF +
CE forward over ~12k music turns × 100 cands).

In [ ]:
# 1) Setup. Disable JAX GPU preallocation BEFORE any import pulls JAX in.
# datasets/transformers import JAX transitively; JAX grabs ~75% of VRAM on
# first use, so the KERNEL ends up hogging the GPU and the cell-3 !python
# subprocess OOMs. This is why nb 72 OOM'd while nb 70/71 (which set these)
# did not. If the kernel already imported JAX, RESTART RUNTIME for this to take.
import os
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')
os.environ.setdefault('TF_FORCE_GPU_ALLOW_GROWTH', 'true')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
from google.colab import userdata, drive
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
drive.mount('/content/drive', force_remount=False)

BRANCH = 'recall-union-lgbm'  # G2: 3-channel union pool + new session features + album_name fix
!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026

DRIVE_BASE = '/content/drive/MyDrive'
LOCAL_BASE = '/content/recsys2026/experiments/cache'
os.makedirs(LOCAL_BASE, exist_ok=True)
src = f'{DRIVE_BASE}/recsys2026_retrieval_v2_cache'
dst = f'{LOCAL_BASE}/retrieval_v2'
if os.path.islink(dst): os.unlink(dst)
elif os.path.exists(dst):
    import shutil; shutil.rmtree(dst)
os.symlink(src, dst)

# Retrieval stack (bm25->bm25s, dense->sentence-transformers/peft) is imported
# eagerly by mcrs.retrieval_modules, so its deps are required even for the
# LGBM build. Matches nb 71's proven set + lightgbm/scikit-learn.
!pip install -q --upgrade 'transformers>=4.40' 'accelerate>=0.30' 'peft>=0.11' \
    'datasets' 'pandas<3.0' 'tqdm' 'huggingface_hub' 'sentence-transformers>=3.0' \
    'FlagEmbedding>=1.3' 'bm25s' 'lightgbm' 'scikit-learn'

In [ ]:
# 2) Walk HF train conversations + session-disjoint 80/20 split.
import sys
sys.path.insert(0, '/content/recsys2026/scripts')
sys.path.insert(0, '/content/recsys2026/music-crs-baselines')
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from build_bi_encoder_training_data import _iter_conversation_turns

train_conv = load_dataset('talkpl-ai/TalkPlayData-Challenge-Dataset', split='train')
all_rows = _iter_conversation_turns(train_conv)
print(f'{len(all_rows)} per-music-turn rows from train split')
session_ids = sorted({r['session_id'] for r in all_rows})
train_sids, val_sids = train_test_split(session_ids, test_size=0.2, random_state=42)
train_set, val_set = set(train_sids), set(val_sids)
train_rows = [r for r in all_rows if r['session_id'] in train_set]
val_rows = [r for r in all_rows if r['session_id'] in val_set]
print(f'train turns: {len(train_rows)}  val turns: {len(val_rows)}')
import json as _j
os.makedirs('experiments/cache/retrieval_v2/lgbm', exist_ok=True)
with open('experiments/cache/retrieval_v2/lgbm/lgbm_train_rows.jsonl', 'w') as f:
    for r in train_rows: f.write(_j.dumps(r, default=str) + '\n')
with open('experiments/cache/retrieval_v2/lgbm/lgbm_val_rows.jsonl', 'w') as f:
    for r in val_rows: f.write(_j.dumps(r, default=str) + '\n')

In [ ]:
# 3) Extract features for each (turn, candidate) pair via Stage A+B pipeline.
# For each music turn:
#   a. Build production query via format_query_text(..., mode='bge_m3_structured').
#   b. wrrf_union_v1 (BM25 + dense_metadata_qwen3 + same_artist) → top-100 candidates.
#   c. extract_features() builds the per-candidate feature row (no cross-encoder).
# Streams live to the cell AND tees a log to Drive; watch the tqdm 'wrrf batches'
# bar for ETA and the final '[lgbm-features] timing: union=... features=...' line.
# Outputs: experiments/cache/retrieval_v2/lgbm/lgbm_{train,val}_features.parquet
# n-sessions=5000 = fast directional run; raise to 999999 for the FINAL model.
!cd /content/recsys2026/music-crs-baselines && python -u ../scripts/build_lgbm_features.py \
    --n-sessions 5000 \
    --topk 100 \
    --seed 42 \
    --out /content/recsys2026/experiments/cache/retrieval_v2/lgbm/lgbm_train_features.parquet \
    --cache-dir /content/recsys2026/experiments/cache \
    2>&1 | tee /content/drive/MyDrive/recsys2026_retrieval_v2_cache/lgbm_build_log.txt
# Note: the existing build_lgbm_features.py samples train sessions; with the
# new 80/20 split, override the sampler by writing a thin per-row driver.
# Implementation detail: pass session_ids filter via the existing --seed +
# n-sessions, OR modify build_lgbm_features.py to accept --session-id-list.
# For Phase 1, the simpler path is: run the full feature extractor on train,
# then post-filter rows to (train_sids, val_sids) into two parquets.

In [ ]:
# 4) Post-filter the single full-train parquet into 80/20 train/val by session.
import pandas as pd
df = pd.read_parquet('/content/recsys2026/experiments/cache/retrieval_v2/lgbm/lgbm_train_features.parquet')
tdf = df[df['session_id'].isin(train_set)].copy()
vdf = df[df['session_id'].isin(val_set)].copy()
tdf.to_parquet('/content/recsys2026/experiments/cache/retrieval_v2/lgbm/lgbm_train_split.parquet', index=False)
vdf.to_parquet('/content/recsys2026/experiments/cache/retrieval_v2/lgbm/lgbm_val_split.parquet', index=False)
print(f'train rows: {len(tdf)}  val rows: {len(vdf)}')
print('positives (label=1):', int(tdf['label'].sum()), int(vdf['label'].sum()))

In [ ]:
# 5) Train LightGBM LambdaRank.
!cd /content/recsys2026 && python -u scripts/train_lgbm_ranker.py \
    --train-features experiments/cache/retrieval_v2/lgbm/lgbm_train_split.parquet \
    --val-features   experiments/cache/retrieval_v2/lgbm/lgbm_val_split.parquet \
    --output-dir     experiments/cache/retrieval_v2/lgbm/lgbm_v1 \
    --n-estimators 1000 \
    2>&1 | tee /content/drive/MyDrive/recsys2026_retrieval_v2_cache/lgbm_train_log.txt
!ls -la /content/recsys2026/experiments/cache/retrieval_v2/lgbm/lgbm_v1/

In [ ]:
# 6) Ensure the trained model is on Drive for inference reuse.
# NOTE: experiments/cache/retrieval_v2 is symlinked to the Drive cache (cell 2),
# so cell 5 already wrote lgbm_v1 ONTO Drive. src and dst below resolve to the
# SAME directory; the old rmtree(dst)+copytree(src) therefore DELETED the model
# and then failed. Guard against that: skip the copy when they're the same path.
import os, shutil
src = '/content/recsys2026/experiments/cache/retrieval_v2/lgbm/lgbm_v1'
dst = '/content/drive/MyDrive/recsys2026_retrieval_v2_cache/lgbm/lgbm_v1'
if not os.path.exists(src):
    raise FileNotFoundError(f'{src} missing — run cell 5 (training) first.')
if os.path.realpath(src) == os.path.realpath(dst):
    print('Model already on Drive via the retrieval_v2 symlink — nothing to copy:')
    print(' ', os.path.realpath(dst))
    print('  contents:', sorted(os.listdir(src)))
else:
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print('LGBM model dir mirrored to:', dst)


In [ ]:
# 7) Dev eval: union (Stage A) -> LGBM rerank (Stage C). NO cross-encoder.
# The old cell loaded a BGE-M3 bi-encoder + a (nonexistent) cross-encoder; that
# pipeline is dropped. This builds dev queries EXACTLY like build_lgbm_features.py
# (raw 'role: content' lines, music turns expanded via id_to_metadata) so the
# query text matches what the LGBM was trained on. Reports dev nDCG@20 for
# union-only vs union+LGBM with a paired-bootstrap CI.
import sys, math
import numpy as np
import pandas as pd
from datasets import load_dataset
sys.path.insert(0, '/content/recsys2026/music-crs-baselines')
from mcrs.db_item.music_catalog import MusicCatalogDB
from mcrs.retrieval_modules import load_retrieval_module
from mcrs.rerankers import load_reranker_module

ITEM_DB   = 'talkpl-ai/TalkPlayData-Challenge-Track-Metadata'
CORPUS    = ['track_name', 'artist_name', 'album_name']
CACHE_DIR = '/content/recsys2026/experiments/cache'
LGBM_DIR  = '/content/recsys2026/experiments/cache/retrieval_v2/lgbm/lgbm_v1'
N_EVAL    = 1000

item_db = MusicCatalogDB(ITEM_DB, ['all_tracks'], CORPUS)

# Build dev queries the SAME way as the training feature builder (raw mode).
dev = load_dataset('talkpl-ai/TalkPlayData-Challenge-Dataset', split='test')
queries, golds, user_ids, played, goal_cats, goal_specs = [], [], [], [], [], []
for sess in dev:
    if len(queries) >= N_EVAL:
        break
    df = pd.DataFrame(sess['conversations'])
    cg = sess.get('conversation_goal') or {}
    for _, music in df[df['role'] == 'music'].iterrows():
        if len(queries) >= N_EVAL:
            break
        turn_n = int(music['turn_number'])
        prior = df[(df['turn_number'] < turn_n) |
                   ((df['turn_number'] == turn_n) & (df['role'] == 'user'))]
        lines = []
        for _, t in prior.iterrows():
            role = 'assistant' if t['role'] == 'music' else t['role']
            content = t['content']
            if t['role'] == 'music':
                try:
                    content = item_db.id_to_metadata(content)
                except Exception:
                    content = str(content)
            lines.append(f'{role}: {content}')
        queries.append(chr(10).join(lines))
        golds.append(music['content'])
        user_ids.append(sess.get('user_id'))
        played.append(list(df[(df['role'] == 'music') & (df['turn_number'] < turn_n)]['content']))
        goal_cats.append(cg.get('category'))
        goal_specs.append(cg.get('specificity'))
print('[dev eval] built', len(queries), 'dev queries')

# Stage A: union top-100. same_artist channel reads ctx['history_tids'].
union = load_retrieval_module('wrrf_union_v1', ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR, extra_config={})
cand100 = union.batch_text_to_item_retrieval(
    queries, topk=100, user_ids=user_ids,
    batch_context=[{'history_tids': p} for p in played])

# Stage C: LGBM rerank top-100 -> top-20. session features read ctx['played_tids'].
lgbm = load_reranker_module('lgbm_rerank', ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR, model_path=LGBM_DIR)
reranked20 = lgbm.rerank(
    queries, cand100, topk=20, user_ids=user_ids,
    goal_categories=goal_cats, goal_specificities=goal_specs,
    extra_session_info=[{'played_tids': p} for p in played])

def ndcg20(ranked, gold):
    r = ranked[:20]
    return 1.0 / math.log2(r.index(gold) + 2) if gold in r else 0.0

a  = np.array([ndcg20(c, g) for c, g in zip(cand100, golds)])     # union only (Stage A)
ac = np.array([ndcg20(r, g) for r, g in zip(reranked20, golds)])  # union + LGBM (A+C)
rec100 = float(np.mean([1.0 if g in c else 0.0 for c, g in zip(cand100, golds)]))

rng = np.random.default_rng(0)
d = ac - a
boot = d[rng.integers(0, len(d), size=(2000, len(d)))].mean(axis=1)
lo, hi = float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))
print('=== Dev nDCG@20 (n=' + str(len(golds)) + ') ===')
print('  union only        :', round(float(a.mean()), 4))
print('  union + LGBM (A+C) :', round(float(ac.mean()), 4))
print('  lift (A+C - A)    :', round(float(d.mean()), 4),
      '95% CI [', round(lo, 4), ',', round(hi, 4), ']',
      'SIGNIFICANT' if lo > 0 else 'not significant')
print('  recall@100 ceiling:', round(rec100, 4))
print('  (Blind-A baseline nDCG@20 = 0.09)')
